# Практическая работа 2. Работа с LLM через API
## Демонстрационный ноутбук: базовое промптирование и извлечение данных

- **Студент:** Григорьев Михаил Романович
- **Группа:** ИСТ.04
- **Вариант:** 26 (`rental_26.csv`)

### 1. Настройка окружения и импорт компонентов

In [1]:
import os
from dotenv import load_dotenv

# Загружаем переменные окружения из .env
load_dotenv()
GIGA_KEY = os.getenv('GIGA_KEY')

# Импорт основных компонентов LangChain
from langchain_gigachat.chat_models import GigaChat
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate
)
from langchain_core.output_parsers import StrOutputParser

### 2. Базовая инициализация модели GigaChat

In [2]:
# Настройка модели GigaChat с расширенными параметрами
llm = GigaChat(
    credentials=GIGA_KEY,
    model='GigaChat-2',
    verify_ssl_certs=False,
    temperature=0.1,
    max_tokens=1000
)

# Проверка подключения
response = llm.invoke('Привет! Как дела?')
print(response.content)

Привет! Всё отлично, спасибо. Готов обсудить что-нибудь интересное или помочь разобраться с задачей. Чем займёмся?


### 3. Системное и пользовательское промптирование

In [3]:
# Шаблон системного сообщения
system_prompt = SystemMessagePromptTemplate.from_template(
    'Ты — эксперт по анализу текстовых заявок на аренду жилья.'
)

# Шаблон пользовательского сообщения
user_prompt = HumanMessagePromptTemplate.from_template(
    'Заявка: {text}\n\nИзвлеки количество человек и верни только число.'
)

# Собираем чат-шаблон
chat_prompt = ChatPromptTemplate.from_messages([system_prompt, user_prompt])

# При вызове цепочки:
chain_chat = chat_prompt | llm | StrOutputParser()
result = chain_chat.invoke({'text': 'Семья из трех человек снимает квартиру'})
print(result) # ожидаем 3

3


### 4. Базовое промптирование для извлечения количества людей (слайд 15)

In [4]:
# Простой промпт для извлечения количества людей
basic_prompt = PromptTemplate(
    input_variables=['text'],
    template="""
Проанализируй следующий текст заявки на аренду жилья и извлеки количество человек, которые будут проживать.
Текст заявки: {text}
Верни только число (целое число), соответствующее количеству проживающих.
Если количество не указано явно, постарайся определить его по контексту.
Количество человек:"""
)

# Создание цепочки
chain = basic_prompt | llm | StrOutputParser()

# Тестирование на типовых примерах
sample_texts = [
    'Ищу квартиру для семьи из четырех человек',
    'Нужна студия для одного',
    'Семья с двумя детьми ищет жилье'
]
for text in sample_texts:
    res = chain.invoke({'text': text})
    print(f'Текст: {text}')
    print(f'Результат: {res}')
    print('---')

Текст: Ищу квартиру для семьи из четырех человек
Результат: 4
---
Текст: Нужна студия для одного
Результат: 1
---
Текст: Семья с двумя детьми ищет жилье
Результат: 4
---


### 5. Индивидуальная работа: тестирование на 15 заявках варианта 26 (слайд 16)

In [5]:
import pandas as pd

# Загрузка 15 заявок варианта 26
df = pd.read_csv('rental_26.csv', sep=';')
test_texts = df['text'].tolist()

print('Результаты обработки 15 заявок:')
for i, text in enumerate(test_texts, 1):
    res = chain.invoke({'text': text})
    print(f'Заявка #{i:02d}: {text[:60]}... -> Результат: {res}')

Заявка #01: Снимем жильё с 1.09по 8.09 двухместный,су в номере ,олимпийс... -> Результат: 2
Заявка #02: Ищем недорогое жилье недалеко от моря. 3-местный и 2-местный... -> Результат: 6
Заявка #03: Здравствуйте,ищем жилье. 2х местный номер,с удобствами. с 17... -> Результат: 2
Заявка #04: Здравствуйте. Интересует жилье 3 местный номер.с20 .06 по 28... -> Результат: 3
Заявка #05: Добрый День!! Семья 4 человека, 2 взрослых, дети 10 и 3 года... -> Результат: 4
Заявка #06: Здравствуйте. Интересует жильё в Лазаревском. 3е взрослых. 
... -> Результат: 3
Заявка #07: Ищем жильё эконом класса, до 1000 на двоих, с 7 по 15 август... -> Результат: 2
Заявка #08: Добрый день. Ищем жильё в Лазаревском. С 10.08.2018 по 26.08... -> Результат: 3
Заявка #09: Здравствуйте! Ищем жильё с 15  по 23 августа 2 взрослых 1 ре... -> Результат: 3
Заявка #10: Добрый день интересует жилье семья 5 человек двое взрослых и... -> Результат: 5
Заявка #11: здравствуйте. с 24 июля по 1 августа ищем жилье для 2-х взро... -> Р